# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [8]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "../datasets/Most Streamed Spotify Songs 2024.csv"


# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [9]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_time
# separo año, mes y día de la fecha de lanzamiento
release_date = pd.to_datetime(spotify_raw['Release Date'])
spotify_raw['year'] = release_date.dt.year
spotify_raw['month'] = release_date.dt.month
spotify_raw['day'] = release_date.dt.day

dim_time = spotify_raw[['year', 'month', 'day']].drop_duplicates().reset_index(drop=True)
dim_time['id_time'] = dim_time.index + 1
display(dim_time.head())

,year,month,day,id_time
0,2024,4,26,1
1,2024,5,4,2
2,2024,3,19,3
3,2023,1,12,4
4,2024,5,31,5


In [10]:
# 2. Crear dim_track
dim_track = spotify_raw[['Track', 'Artist', 'Explicit Track']].drop_duplicates().reset_index(drop=True)
dim_track['id_track'] = dim_track.index + 1
display(dim_track.head())

,Track,Artist,Explicit Track,id_track
0,MILLION DOLLAR BABY,Tommy Richman,0,1
1,Not Like Us,Kendrick Lamar,1,2
2,i like the way you kiss me,Artemas,0,3
3,Flowers,Miley Cyrus,0,4
4,Houdini,Eminem,1,5


In [11]:
# 3. Crear fact_streams
# los streams y views vienen como texto con comas ("390,470,936"), los paso a número para poder sumarlos
for col in ['Spotify Streams', 'YouTube Views']:
    spotify_raw[col] = pd.to_numeric(spotify_raw[col].str.replace(',', ''))

fact_streams = spotify_raw.merge(dim_time, on=['year', 'month', 'day']).merge(dim_track, on=['Track', 'Artist', 'Explicit Track'])
fact_streams = fact_streams[['id_time', 'id_track', 'Spotify Streams', 'YouTube Views', 'Track Score']]
display(fact_streams.head())

,id_time,id_track,Spotify Streams,YouTube Views,Track Score
0,1,1,3.904709e+08,8.427475e+07,725.4
1,2,2,3.237039e+08,1.163470e+08,545.9
2,3,3,6.013093e+08,1.225991e+08,538.4
3,4,4,2.031281e+09,1.096101e+09,444.9
4,5,5,1.070349e+08,7.737396e+07,423.3


In [12]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
olap_cube = fact_streams.merge(dim_time, on='id_time').merge(dim_track, on='id_track')
display(olap_cube.head())

,id_time,id_track,Spotify Streams,YouTube Views,Track Score,year,month,day,Track,Artist,Explicit Track
0,1,1,3.904709e+08,8.427475e+07,725.4,2024,4,26,MILLION DOLLAR BABY,Tommy Richman,0
1,2,2,3.237039e+08,1.163470e+08,545.9,2024,5,4,Not Like Us,Kendrick Lamar,1
2,3,3,6.013093e+08,1.225991e+08,538.4,2024,3,19,i like the way you kiss me,Artemas,0
3,4,4,2.031281e+09,1.096101e+09,444.9,2023,1,12,Flowers,Miley Cyrus,0
4,5,5,1.070349e+08,7.737396e+07,423.3,2024,5,31,Houdini,Eminem,1


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

Hay 2 porque son lo mínimo que piden las consultas, todo se filtra o agrupa por fecha o por canción/artista. Pero sí se podrían agregar más, por ejemplo una dimensión de álbum separar al artista en su propia dimensión, ya que un artista tiene muchas canciones y así no se repite su nombre en cada pista.

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [13]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: Roll-up
answer_1 = olap_cube.groupby('Artist')['Spotify Streams'].sum().sort_values(ascending=False)
display(answer_1.head(10))

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: Slice
answer_2 = olap_cube[olap_cube['year'] == 2024]
display(answer_2.head())

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: Drill-down
answer_3 = olap_cube.groupby(['year', 'month'])['Spotify Streams'].sum()
display(answer_3)

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: Dice
answer_4 = olap_cube[(olap_cube['Explicit Track'] == 1) & (olap_cube['year'] == 2024)]
display(answer_4.head())

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: Roll-up
answer_5 = olap_cube.groupby('year')['Track Score'].mean()
display(answer_5)

Artist
Bad Bunny         3.705483e+10
The Weeknd        3.694854e+10
Drake             3.496216e+10
Taylor Swift      3.447077e+10
Post Malone       2.613747e+10
Ed Sheeran        2.401490e+10
Ariana Grande     2.346499e+10
MUSIC LAB JPN     2.286669e+10
Olivia Rodrigo    1.972922e+10
Eminem            1.887888e+10
Name: Spotify Streams, dtype: float64

,id_time,id_track,Spotify Streams,YouTube Views,Track Score,year,month,day,Track,Artist,Explicit Track
0,1,1,390470936.0,84274754.0,725.4,2024,4,26,MILLION DOLLAR BABY,Tommy Richman,0
1,2,2,323703884.0,116347040.0,545.9,2024,5,4,Not Like Us,Kendrick Lamar,1
2,3,3,601309283.0,122599116.0,538.4,2024,3,19,i like the way you kiss me,Artemas,0
4,5,5,107034922.0,77373957.0,423.3,2024,5,31,Houdini,Eminem,1
6,7,7,900158751.0,308723145.0,407.2,2024,1,18,Beautiful Things,Benson Boone,0


year  month
1987  7        1.879386e+09
1991  9        2.021910e+09
1994  1        1.810650e+09
1998  4        1.783399e+08
1999  6        1.405354e+09
                   ...     
2024  2        1.023077e+10
      3        1.263673e+10
      4        2.152035e+10
      5        1.005348e+10
      6        2.711085e+09
Name: Spotify Streams, Length: 191, dtype: float64

,id_time,id_track,Spotify Streams,YouTube Views,Track Score,year,month,day,Track,Artist,Explicit Track
1,2,2,3.237039e+08,116347040.0,545.9,2024,5,4,Not Like Us,Kendrick Lamar,1
4,5,5,1.070349e+08,77373957.0,423.3,2024,5,31,Houdini,Eminem,1
7,8,8,6.750792e+08,228382568.0,375.8,2024,2,2,Gata Only,FloyyMenor,1
8,9,9,1.653018e+09,NaN,355.7,2024,6,9,Danza Kuduro - Cover,MUSIC LAB JPN,1
9,10,10,9.067657e+07,32735244.0,330.6,2024,5,23,BAND4BAND (feat. Lil Baby),Central Cee,1


year
1987    21.500000
1991    19.600000
1994    31.500000
1998    20.800000
1999    22.200000
2000    30.400000
2001    20.100000
2002    26.060000
2003    23.766667
2004    24.240000
2005    27.140000
2006    22.300000
2007    23.860000
2008    23.854545
2009    26.000000
2010    31.425926
2011    31.800000
2012    32.197872
2013    31.674545
2014    31.678571
2015    37.988542
2016    37.828455
2017    37.949765
2018    36.977510
2019    39.380405
2020    38.042222
2021    38.144118
2022    41.693228
2023    43.264594
2024    54.120058
Name: Track Score, dtype: float64

In [14]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: Slice
answer_6 = olap_cube[olap_cube['Artist'] == 'Bad Bunny']
display(answer_6.head())

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: Drill-down
answer_7 = olap_cube.groupby(['Artist', 'Track'])['YouTube Views'].sum()
display(answer_7)

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: Dice
answer_8 = olap_cube[(olap_cube['year'] == 2023) & (olap_cube['month'] == 12)]
display(answer_8.head())

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: Roll-up
answer_9 = olap_cube.groupby('month')['Spotify Streams'].sum().sort_values(ascending=False)
display(answer_9)

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: Dice
answer_10 = olap_cube[(olap_cube['Artist'] == 'Taylor Swift') & (olap_cube['Explicit Track'] == 0) & (olap_cube['year'] == 2024)]
display(answer_10)

,id_time,id_track,Spotify Streams,YouTube Views,Track Score,year,month,day,Track,Artist,Explicit Track
63,57,61,1.811991e+09,1.516863e+09,184.8,2022,5,6,Me Porto Bonito,Bad Bunny,1
88,57,86,1.514240e+09,1.366181e+09,163.1,2022,5,6,Titï¿½ï¿½ Me Pregu,Bad Bunny,1
102,91,100,4.781509e+08,1.133797e+08,150.7,2023,10,13,MONACO,Bad Bunny,1
174,91,171,6.891815e+08,1.833815e+08,114.2,2023,10,13,PERRO NEGRO,Bad Bunny,1
187,57,184,1.421763e+09,9.458571e+08,110.7,2022,5,6,Ojitos Lindos,Bad Bunny,0


Artist                                       Track                                                              
"XY"                                         Come and Say - Original Mix                                            415159148.0
$OHO BANI                                    ZEIT, DASS SICH WAS DREHT                                                6744201.0
$uicideboy$                                  ...And to Those I Love, Thanks for Sticking Around                     269460083.0
                                             1000 Blunts                                                             87680985.0
                                             Are You Going to See the Rose in the Vase, or the Dust on the Table      2640169.0
                                                                                                                       ...     
ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý              ýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýý                              44

,id_time,id_track,Spotify Streams,YouTube Views,Track Score,year,month,day,Track,Artist,Explicit Track
61,55,59,1384.0,222076.0,186.5,2023,12,31,Very Cute Melody by Marimba Tone (39813),mitsu sound,0
94,84,92,649404552.0,433839212.0,155.9,2023,12,2,La Diabla,Xavi,0
97,87,95,359803580.0,369321334.0,153.1,2023,12,7,BELLAKEO,Peso Pluma,1
131,112,129,8327.0,NaN,130.7,2023,12,10,Looking Like a Wow End of Year 2023,WasteLand,0
177,152,174,606124553.0,166363445.0,112.6,2023,12,1,LUNA,Feid,0


month
1     3.652168e+11
5     1.779293e+11
11    1.734614e+11
10    1.672667e+11
3     1.638041e+11
7     1.503832e+11
6     1.484293e+11
4     1.462846e+11
8     1.426826e+11
9     1.363626e+11
2     1.333565e+11
12    1.022498e+11
Name: Spotify Streams, dtype: float64

,id_time,id_track,Spotify Streams,YouTube Views,Track Score,year,month,day,Track,Artist,Explicit Track
16,17,17,395433400.0,104673836.0,297.6,2024,4,18,Fortnight (feat. Post Malone),Taylor Swift,0
866,17,861,166168092.0,16423009.0,52.3,2024,4,18,Whoï¿½ï¿½ï¿½s Afraid of Little O,Taylor Swift,0
1013,17,1007,175613480.0,15237337.0,47.8,2024,4,18,"So Long, London",Taylor Swift,0
1167,17,1159,161916445.0,13509432.0,44.0,2024,4,18,My Boy Only Breaks His Favorite Toys,Taylor Swift,0
1270,17,1260,165611453.0,9873283.0,42.1,2024,4,18,Guilty as Sin?,Taylor Swift,0
2484,183,2454,97601615.0,5528058.0,28.6,2024,4,19,So High School,Taylor Swift,0
2517,17,2486,106201248.0,5521347.0,28.3,2024,4,18,The Alchemy,Taylor Swift,0
2528,17,2497,250430303.0,NaN,28.2,2024,4,18,I Can Do It With a Broken Heart,Taylor Swift,0
2776,183,2744,88244010.0,7381186.0,26.7,2024,4,19,imgonnagetyouback,Taylor Swift,0
2816,17,2784,105424914.0,7227379.0,26.5,2024,4,18,Fresh Out The Slammer,Taylor Swift,0
